In [1]:
# Step 1: Setup & Load Data
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv("tripadvisor_hotel_reviews.csv")

# Rebuild clauses (same logic as EDA)
def split_into_clauses(text, min_words=5):
    text = str(text).strip()
    clauses = re.split(r'[,\.!\?]+', text)
    clauses = [c.strip() for c in clauses if len(c.strip().split()) >= min_words]
    return clauses

df['clauses'] = df['Review'].apply(split_into_clauses)

print(f"Reviews loaded: {len(df):,}")
print(f"Total clauses: {df['clauses'].apply(len).sum():,}")
print("Setup complete ✅")


Reviews loaded: 20,491
Total clauses: 203,729
Setup complete ✅


In [3]:
# Step 2: Define Labeling Rules — Keyword-based Weak Supervision

RULES = {
    'cleanliness': {
        'triggers':  ['clean', 'dirty', 'spotless', 'filthy', 'hygiene', 'hygienic',
                      'dust', 'dusty', 'stain', 'stained', 'smell', 'smelly', 'odor',
                      'odour', 'gross', 'immaculate', 'sanitiz', 'tidy', 'messy',
                      'mold', 'mould', 'pest', 'bug', 'cockroach'],
        'positive':  ['clean', 'spotless', 'immaculate', 'tidy', 'fresh', 'sanitiz',
                      'hygienic', 'pristine', 'sparkling'],
        'negative':  ['dirty', 'filthy', 'dusty', 'stain', 'stained', 'smelly', 'odor',
                      'odour', 'gross', 'messy', 'mold', 'mould', 'pest', 'bug',
                      'cockroach', 'disgusting', 'unhygienic']
    },
    'staff_service': {
        'triggers':  ['staff', 'service', 'front desk', 'concierge', 'receptionist',
                      'housekeep', 'bellm', 'valet', 'employee', 'manager', 'host',
                      'check in', 'check-in', 'checkout', 'check out', 'reception'],
        'positive':  ['friendly', 'helpful', 'attentive', 'professional', 'excellent service',
                      'great service', 'wonderful', 'courteous', 'polite', 'accommodating',
                      'responsive', 'amazing staff', 'fantastic staff', 'nice staff',
                      'kind', 'warm', 'efficient', 'quick', 'pleasant'],
        'negative':  ['rude', 'unhelpful', 'unfriendly', 'slow', 'ignorant', 'ignored',
                      'indifferent', 'dismissive', 'poor service', 'bad service',
                      'terrible service', 'incompetent', 'unprofessional', 'arrogant',
                      'snooty', 'not helpful', 'did not help', 'no help']
    },
    'wifi_quality': {
        'triggers':  ['wifi', 'wi-fi', 'internet', 'wireless', 'connection', 'online',
                      'bandwidth', 'signal', 'broadband', 'network'],
        'positive':  ['fast', 'good wifi', 'great wifi', 'excellent wifi', 'strong signal',
                      'reliable', 'good internet', 'fast internet', 'free wifi',
                      'complimentary wifi', 'worked well', 'no problem', 'speedy'],
        'negative':  ['slow', 'bad wifi', 'poor wifi', 'weak signal', 'no wifi',
                      'no internet', 'terrible wifi', 'not working', 'unreliable',
                      'poor connection', 'kept dropping', 'expensive',
                      'pay for wifi', 'charged for wifi', 'patchy']
    },
    'noise_level': {
        'triggers':  ['noise', 'noisy', 'loud', 'quiet', 'sound', 'soundproof',
                      'hear', 'heard', 'bang', 'traffic', 'peaceful', 'disturb',
                      'thin walls', 'wall', 'neighbor', 'street noise'],
        'positive':  ['quiet', 'peaceful', 'silent', 'soundproof', 'tranquil',
                      'calm', 'no noise', 'not noisy', 'good sleep'],
        'negative':  ['noisy', 'loud', 'disturbing', 'disturbed', 'kept awake',
                      'could hear', 'thin walls', 'street noise', 'traffic noise',
                      'loud neighbors', 'banging', 'construction', 'not soundproof',
                      'too loud', 'heard everything']
    },
    'location': {
        'triggers':  ['location', 'located', 'walk', 'distance', 'nearby', 'transit',
                      'transport', 'central', 'close to', 'far from', 'neighborhood',
                      'area', 'downtown', 'access', 'convenient', 'commute', 'metro',
                      'bus', 'train', 'airport', 'station'],
        'positive':  ['great location', 'perfect location', 'excellent location',
                      'central location', 'convenient', 'walking distance', 'close to',
                      'easy access', 'well located', 'ideal location', 'good area',
                      'safe area', 'near', 'accessible', 'good transport'],
        'negative':  ['bad location', 'poor location', 'far from', 'inconvenient',
                      'hard to reach', 'unsafe area', 'sketchy', 'remote',
                      'not central', 'difficult access', 'far away', 'too far']
    }
}

print("Labeling rules defined ✅")
print(f"\nAttributes: {list(RULES.keys())}")
for attr, rule in RULES.items():
    print(f"  {attr:<20} triggers={len(rule['triggers'])} | pos={len(rule['positive'])} | neg={len(rule['negative'])}")


Labeling rules defined ✅

Attributes: ['cleanliness', 'staff_service', 'wifi_quality', 'noise_level', 'location']
  cleanliness          triggers=24 | pos=9 | neg=17
  staff_service        triggers=16 | pos=19 | neg=18
  wifi_quality         triggers=10 | pos=13 | neg=15
  noise_level          triggers=16 | pos=9 | neg=15
  location             triggers=21 | pos=15 | neg=12


In [4]:
# Step 3: Apply Labels to All Clauses

def label_clause(clause, rating, rules):
    """
    Label a single clause for all 5 attributes.
    Returns dict: {attr: 'positive'/'negative'/'not_mentioned'}
    """
    clause_lower = clause.lower()
    labels = {}

    for attr, rule in rules.items():
        # Check if any trigger keyword is present
        triggered = any(kw in clause_lower for kw in rule['triggers'])

        if not triggered:
            labels[attr] = 'not_mentioned'
            continue

        # Check polarity keywords
        has_positive = any(kw in clause_lower for kw in rule['positive'])
        has_negative = any(kw in clause_lower for kw in rule['negative'])

        if has_positive and has_negative:
            # Conflict: use rating as tiebreaker
            labels[attr] = 'positive' if rating >= 4 else 'negative'
        elif has_positive:
            labels[attr] = 'positive'
        elif has_negative:
            labels[attr] = 'negative'
        else:
            # Trigger present but no polarity — use rating as weak signal
            if rating >= 4:
                labels[attr] = 'positive'
            elif rating <= 2:
                labels[attr] = 'negative'
            else:
                labels[attr] = 'not_mentioned'  # rating 3 = ambiguous

    return labels


# Build flat clause-level dataset
print("Labeling all clauses... (may take ~30 seconds)")
rows = []
for idx, row in df.iterrows():
    for clause in row['clauses']:
        label_dict = label_clause(clause, row['Rating'], RULES)
        rows.append({
            'review_idx': idx,
            'rating': row['Rating'],
            'clause': clause,
            **label_dict
        })

labeled_df = pd.DataFrame(rows)
print(f"Done! Total labeled clauses: {len(labeled_df):,}")
print(f"\nColumns: {labeled_df.columns.tolist()}")
print("\nSample rows:")
labeled_df.head(3)


Labeling all clauses... (may take ~30 seconds)
Done! Total labeled clauses: 203,729

Columns: ['review_idx', 'rating', 'clause', 'cleanliness', 'staff_service', 'wifi_quality', 'noise_level', 'location']

Sample rows:


,review_idx,rating,clause,cleanliness,staff_service,wifi_quality,noise_level,location
0,0,4,nice hotel expensive parking got good deal sta...,not_mentioned,not_mentioned,not_mentioned,not_mentioned,not_mentioned
1,0,4,arrived late evening took advice previous revi...,not_mentioned,positive,not_mentioned,not_mentioned,not_mentioned
2,0,4,little disappointed non-existent view room roo...,positive,not_mentioned,not_mentioned,not_mentioned,not_mentioned


In [5]:
# Step 3b: View actual labeled clauses (non-trivial labels)

attrs = ['cleanliness', 'staff_service', 'wifi_quality', 'noise_level', 'location']

for attr in attrs:
    print(f"\n{'='*60}")
    print(f"  ATTRIBUTE: {attr.upper()}")
    print(f"{'='*60}")

    pos_examples = labeled_df[labeled_df[attr] == 'positive']['clause'].head(3).tolist()
    neg_examples = labeled_df[labeled_df[attr] == 'negative']['clause'].head(3).tolist()

    print("  ✅ POSITIVE examples:")
    for i, ex in enumerate(pos_examples, 1):
        print(f"    [{i}] {ex[:120]}")

    print("  ❌ NEGATIVE examples:")
    for i, ex in enumerate(neg_examples, 1):
        print(f"    [{i}] {ex[:120]}")



  ATTRIBUTE: CLEANLINESS
  ✅ POSITIVE examples:
    [1] little disappointed non-existent view room room clean nice size
    [2] arrival no champagne strawberries no foam pillows great room view alley high rise building good not better housekeeping 
    [3] this not 4 start hotel clean business hotel super high rates
  ❌ NEGATIVE examples:
    [1] bed linen torn curtains torn floor stains
    [2] warwick mats garage filthy stairwells
    [3] hotel faces 4th smells urine

  ATTRIBUTE: STAFF_SERVICE
  ✅ POSITIVE examples:
    [1] arrived late evening took advice previous reviews did valet parking
    [2] positives large bathroom mediterranean suite comfortable bed pillowsattentive housekeeping staffnegatives ac unit malfun
    [3] did n't partake free wine coffee/tea service lobby thought great feature
  ❌ NEGATIVE examples:
    [1] send kimpton preferred guest website email asking failure provide suite advertised website reservation description furni
    [2] the staff ranged indifferent

In [6]:
# Step 4: Label Distribution Analysis
attrs = ['cleanliness', 'staff_service', 'wifi_quality', 'noise_level', 'location']

print("=== LABEL DISTRIBUTION PER ATTRIBUTE ===\n")
print(f"{'Attribute':<20} {'positive':>10} {'negative':>10} {'not_mentioned':>15} {'total_labeled':>15}")
print("-" * 72)

for attr in attrs:
    counts = labeled_df[attr].value_counts()
    pos = counts.get('positive', 0)
    neg = counts.get('negative', 0)
    nm  = counts.get('not_mentioned', 0)
    total = pos + neg
    print(f"{attr:<20} {pos:>10,} {neg:>10,} {nm:>15,} {total:>15,}")

print(f"\n{'TOTAL CLAUSES':<20} {len(labeled_df):>10,}")

# Positive ratio per attribute
print("\n=== POSITIVE RATE (pos / pos+neg) ===")
for attr in attrs:
    counts = labeled_df[attr].value_counts()
    pos = counts.get('positive', 0)
    neg = counts.get('negative', 0)
    total = pos + neg
    if total > 0:
        print(f"  {attr:<20} {pos/(pos+neg)*100:.1f}% positive  ({total:,} labeled clauses)")


=== LABEL DISTRIBUTION PER ATTRIBUTE ===

Attribute              positive   negative   not_mentioned   total_labeled
------------------------------------------------------------------------
cleanliness               9,618      2,638         191,473          12,256
staff_service            19,648      5,091         178,990          24,739
wifi_quality              1,950        514         201,265           2,464
noise_level               6,574      3,307         193,848           9,881
location                 28,698      3,748         171,283          32,446

TOTAL CLAUSES           203,729

=== POSITIVE RATE (pos / pos+neg) ===
  cleanliness          78.5% positive  (12,256 labeled clauses)
  staff_service        79.4% positive  (24,739 labeled clauses)
  wifi_quality         79.1% positive  (2,464 labeled clauses)
  noise_level          66.5% positive  (9,881 labeled clauses)
  location             88.4% positive  (32,446 labeled clauses)


In [7]:
# Step 5: Quality Spot Check
import random
random.seed(42)

attrs = ['cleanliness', 'staff_service', 'wifi_quality', 'noise_level', 'location']

for attr in attrs:
    print(f"\n{'='*65}")
    print(f"  SPOT CHECK: {attr.upper()}")
    print(f"{'='*65}")
    for label in ['positive', 'negative']:
        subset = labeled_df[labeled_df[attr] == label]['clause'].tolist()
        samples = random.sample(subset, min(5, len(subset)))
        print(f"\n  [{label.upper()}]")
        for s in samples:
            print(f"    → {s[:110]}")



  SPOT CHECK: CLEANLINESS

  [POSITIVE]
    → like embassy suites rooms clean staff attentive
    → staff nice rooms clean thing room shower no tub
    → location excellent room clean breakfast good staff n't helpful
    → rooms 6th floor clean spacious paris standards fantastic view eiffel tower
    → the room spacious clean maintained stylish

  [NEGATIVE]
    → while wife kept waking morning looked like bug bites upper leg
    → homework book bavaro princess don__Ç_é_ care cockroaches mosquitoes floods suite bavaro princes destination
    → old sink dirty crust soap things shared shower disaster really freaked inside second time
    → hotel packed people felt run dirty did n't want boat forth access beach everyday
    → we stayed room overnight requested room change day staff not accomidating stating mean room smells

  SPOT CHECK: STAFF_SERVICE

  [POSITIVE]
    → doormen provided excellent service friendly
    → staff friendly helpful committed making feel welcome comfortable eag

In [8]:
# Step 6: Patch obvious errors + Export

# Fix 1: Add missing negative keywords for wifi
RULES['wifi_quality']['negative'].extend(['worst', 'expensive', 'costly', 'high cost', 'overpriced'])

# Fix 2: Remove overly generic location triggers
RULES['location']['triggers'] = [
    'location', 'located', 'walking distance', 'distance from', 'nearby',
    'transit', 'transport', 'central location', 'close to', 'far from',
    'neighborhood', 'downtown', 'convenient location', 'commute',
    'metro', 'train station', 'bus stop', 'airport', 'attraction'
]

# Fix 3: Remove 'neighbor' from noise (matches 'neighborhood' ambiguously)
RULES['noise_level']['triggers'] = [
    'noise', 'noisy', 'loud', 'quiet', 'soundproof', 'hear', 'heard',
    'banging', 'traffic noise', 'street noise', 'peaceful', 'disturb',
    'thin wall', 'next door', 'construction'
]

# Re-label with fixes
print("Re-labeling with patched rules...")
rows_final = []
for idx, row in df.iterrows():
    for clause in row['clauses']:
        label_dict = label_clause_v2(clause, row['Rating'], RULES)
        rows_final.append({
            'review_idx': idx,
            'rating': row['Rating'],
            'clause': clause,
            **label_dict
        })

labeled_df = pd.DataFrame(rows_final)
print(f"Done! Total clauses: {len(labeled_df):,}")

# Export
labeled_df.to_csv('data/labeled_clauses.csv', index=False)
print("Exported → data/labeled_clauses.csv ✅")

# Final distribution
print("\n=== FINAL LABEL DISTRIBUTION ===")
attrs = ['cleanliness', 'staff_service', 'wifi_quality', 'noise_level', 'location']
print(f"{'Attribute':<20} {'positive':>10} {'negative':>10} {'labeled_total':>15}")
print("-" * 57)
for attr in attrs:
    counts = labeled_df[attr].value_counts()
    pos = counts.get('positive', 0)
    neg = counts.get('negative', 0)
    print(f"{attr:<20} {pos:>10,} {neg:>10,} {pos+neg:>15,}")


Re-labeling with patched rules...


NameError: name 'label_clause_v2' is not defined

In [9]:
# Install transformers
import subprocess
subprocess.run(['pip', 'install', 'transformers', 'torch'], check=True)
print("Done ✅")


Done ✅


In [10]:
# Load sentiment model (trained on hotel/restaurant reviews)
from transformers import pipeline

print("Loading sentiment model... (first run downloads ~700MB, be patient ⏳)")
sentiment_model = pipeline(
    "text-classification",
    model="nlptown/bert-base-multilingual-uncased-sentiment",
    top_k=1
)
print("Model loaded ✅")

# Quick accuracy test
test_clauses = [
    "room was spotless and immaculate",
    "worst high cost internet access completely unusable",
    "staff were incredibly rude and unhelpful",
    "perfect location walking distance to everything",
    "very noisy construction outside all night",
    "free wifi worked great no issues at all",
    "bed comfortable woke stiff neck high pillows"
]
print("\n=== MODEL TEST ===")
for clause in test_clauses:
    result = sentiment_model(clause)[0][0]
    stars = int(result['label'].split()[0])
    polarity = "positive" if stars >= 4 else ("negative" if stars <= 2 else "neutral")
    print(f"  [{polarity:8}] ({stars}★ | {result['score']:.2f}) → {clause}")


Loading sentiment model... (first run downloads ~700MB, be patient ⏳)


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 6706.08it/s]


Model loaded ✅

=== MODEL TEST ===
  [negative] (1★ | 0.54) → room was spotless and immaculate
  [negative] (1★ | 0.92) → worst high cost internet access completely unusable
  [negative] (1★ | 0.70) → staff were incredibly rude and unhelpful
  [positive] (5★ | 0.79) → perfect location walking distance to everything
  [negative] (1★ | 0.58) → very noisy construction outside all night
  [positive] (5★ | 0.75) → free wifi worked great no issues at all
  [positive] (4★ | 0.46) → bed comfortable woke stiff neck high pillows


In [11]:
# Switch to DistilBERT SST-2 (better for short lowercased clauses)
from transformers import pipeline

print("Loading DistilBERT sentiment model...")
sentiment_model = pipeline(
    "text-classification",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    top_k=1
)
print("Model loaded ✅")

# Re-run the same test
test_clauses = [
    "room was spotless and immaculate",
    "worst high cost internet access completely unusable",
    "staff were incredibly rude and unhelpful",
    "perfect location walking distance to everything",
    "very noisy construction outside all night",
    "free wifi worked great no issues at all",
    "bed comfortable woke stiff neck high pillows",
    "not soundproof heard music room night loud bangs doors",
    "rooms clean modern hotel quiet elegance"
]

print("\n=== MODEL TEST ===")
for clause in test_clauses:
    result = sentiment_model(clause)[0][0]
    label = result['label'].lower()  # 'positive' or 'negative'
    score = result['score']
    print(f"  [{label:8}] ({score:.2f}) → {clause}")


Loading DistilBERT sentiment model...


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 6204.86it/s]


Model loaded ✅

=== MODEL TEST ===
  [positive] (1.00) → room was spotless and immaculate
  [negative] (1.00) → worst high cost internet access completely unusable
  [negative] (1.00) → staff were incredibly rude and unhelpful
  [positive] (1.00) → perfect location walking distance to everything
  [negative] (1.00) → very noisy construction outside all night
  [negative] (0.89) → free wifi worked great no issues at all
  [negative] (0.91) → bed comfortable woke stiff neck high pillows
  [negative] (1.00) → not soundproof heard music room night loud bangs doors
  [positive] (1.00) → rooms clean modern hotel quiet elegance


In [12]:
# Step 7: Final Hybrid Labeler (keyword trigger + DistilBERT polarity)

def label_clause_hybrid(clause, rules, sentiment_model):
    """
    For each attribute:
    - Use keyword rules to detect if attribute is mentioned (trigger)
    - Use DistilBERT to determine polarity (positive/negative)
    - If no trigger → not_mentioned
    """
    clause_lower = clause.lower()
    labels = {}

    for attr, rule in rules.items():
        # Word-boundary trigger check
        triggered = any(contains_keyword(clause_lower, kw) for kw in rule['triggers'])

        if not triggered:
            labels[attr] = 'not_mentioned'
            continue

        # Use sentiment model for polarity
        try:
            result = sentiment_model(clause[:512])[0][0]  # truncate for safety
            sentiment = result['label'].lower()  # 'positive' or 'negative'
            labels[attr] = sentiment
        except Exception:
            labels[attr] = 'not_mentioned'

    return labels


# Apply to ALL clauses (this will take 10-20 mins on CPU — batched for speed)
from tqdm.notebook import tqdm

# Collect all triggered clauses first (avoid running model on not_mentioned)
print("Collecting triggered clauses...")
all_rows = []
triggered_indices = []   # (review_idx, clause_idx, attr)

for idx, row in df.iterrows():
    for clause in row['clauses']:
        all_rows.append({
            'review_idx': idx,
            'rating': row['Rating'],
            'clause': clause,
        })

clause_df = pd.DataFrame(all_rows)
print(f"Total clauses: {len(clause_df):,}")

# Pre-check triggers for each attribute
attrs = list(RULES.keys())
for attr in attrs:
    triggers = RULES[attr]['triggers']
    clause_df[f'trigger_{attr}'] = clause_df['clause'].apply(
        lambda c: any(contains_keyword(c.lower(), kw) for kw in triggers)
    )
    triggered_count = clause_df[f'trigger_{attr}'].sum()
    print(f"  {attr:<20}: {triggered_count:,} triggered clauses")

print("\nTrigger detection done ✅")


Total clauses: 203,729


NameError: name 'contains_keyword' is not defined

In [13]:
# Step 7 (Self-contained): All dependencies + Trigger Detection

import re
import pandas as pd

# --- Helper functions ---
def contains_keyword(text, keyword):
    pattern = r'\b' + re.escape(keyword) + r'\b'
    return bool(re.search(pattern, text, re.IGNORECASE))

# --- Updated RULES ---
RULES = {
    'cleanliness': {
        'triggers': ['clean', 'dirty', 'spotless', 'filthy', 'hygiene', 'hygienic',
                     'dust', 'dusty', 'stain', 'stained', 'smell', 'smelly', 'odor',
                     'odour', 'gross', 'immaculate', 'sanitiz', 'tidy', 'messy',
                     'mold', 'mould', 'pest', 'cockroach'],
    },
    'staff_service': {
        'triggers': ['staff', 'service', 'front desk', 'concierge', 'receptionist',
                     'housekeep', 'bellm', 'valet', 'employee', 'manager',
                     'check in', 'check-in', 'checkout', 'check out', 'reception'],
    },
    'wifi_quality': {
        'triggers': ['wifi', 'wi-fi', 'internet', 'wireless', 'bandwidth', 'broadband'],
    },
    'noise_level': {
        'triggers': ['noise', 'noisy', 'loud', 'quiet', 'soundproof', 'hear', 'heard',
                     'banging', 'traffic noise', 'street noise', 'peaceful', 'disturb',
                     'thin wall', 'construction'],
    },
    'location': {
        'triggers': ['location', 'located', 'walking distance', 'distance from', 'nearby',
                     'transit', 'transport', 'central location', 'close to', 'far from',
                     'neighborhood', 'downtown', 'convenient location',
                     'metro', 'train station', 'bus stop', 'airport', 'attraction'],
    }
}

# --- Rebuild clause dataframe ---
def split_into_clauses(text, min_words=5):
    text = str(text).strip()
    clauses = re.split(r'[,\.!\?]+', text)
    return [c.strip() for c in clauses if len(c.strip().split()) >= min_words]

df['clauses'] = df['Review'].apply(split_into_clauses)

all_rows = []
for idx, row in df.iterrows():
    for clause in row['clauses']:
        all_rows.append({'review_idx': idx, 'rating': row['Rating'], 'clause': clause})

clause_df = pd.DataFrame(all_rows)
print(f"Total clauses: {len(clause_df):,}")

# --- Trigger detection per attribute ---
attrs = list(RULES.keys())
for attr in attrs:
    triggers = RULES[attr]['triggers']
    clause_df[f'trigger_{attr}'] = clause_df['clause'].apply(
        lambda c: any(contains_keyword(c.lower(), kw) for kw in triggers)
    )
    print(f"  {attr:<20}: {clause_df[f'trigger_{attr}'].sum():,} triggered clauses")

print("\nTrigger detection done ✅")


Total clauses: 203,729
  cleanliness         : 9,589 triggered clauses
  staff_service       : 24,364 triggered clauses
  wifi_quality        : 2,271 triggered clauses
  noise_level         : 7,255 triggered clauses
  location            : 17,457 triggered clauses

Trigger detection done ✅


In [14]:
# Step 8: Apply DistilBERT to all triggered clauses (batched)
from tqdm.notebook import tqdm
import numpy as np

# Get all unique triggered clauses (across all attributes)
triggered_mask = clause_df[[f'trigger_{a}' for a in attrs]].any(axis=1)
triggered_clauses = clause_df[triggered_mask]['clause'].unique().tolist()
print(f"Unique triggered clauses to score: {len(triggered_clauses):,}")

# Run model in batches of 64
BATCH_SIZE = 64
sentiment_cache = {}  # clause_text → 'positive' or 'negative'

print("Running DistilBERT sentiment inference...")
for i in tqdm(range(0, len(triggered_clauses), BATCH_SIZE)):
    batch = triggered_clauses[i:i+BATCH_SIZE]
    # Truncate each clause to 512 chars
    batch_truncated = [c[:512] for c in batch]
    results = sentiment_model(batch_truncated, batch_size=BATCH_SIZE)
    for clause, result in zip(batch, results):
        label = result[0]['label'].lower()  # 'positive' or 'negative'
        sentiment_cache[clause] = label

print(f"\nDone! Scored {len(sentiment_cache):,} unique clauses ✅")

# Quick check on cache
pos_count = sum(1 for v in sentiment_cache.values() if v == 'positive')
neg_count = sum(1 for v in sentiment_cache.values() if v == 'negative')
print(f"  Positive: {pos_count:,} ({pos_count/len(sentiment_cache)*100:.1f}%)")
print(f"  Negative: {neg_count:,} ({neg_count/len(sentiment_cache)*100:.1f}%)")


Unique triggered clauses to score: 54,311
Running DistilBERT sentiment inference...


ImportError: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html

In [15]:
# Step 8 (Fixed): Apply DistilBERT to all triggered clauses
from tqdm import tqdm  # ← regular tqdm, not notebook version

# Get all unique triggered clauses (across all attributes)
triggered_mask = clause_df[[f'trigger_{a}' for a in attrs]].any(axis=1)
triggered_clauses = clause_df[triggered_mask]['clause'].unique().tolist()
print(f"Unique triggered clauses to score: {len(triggered_clauses):,}")

# Run model in batches of 64
BATCH_SIZE = 64
sentiment_cache = {}  # clause_text → 'positive' or 'negative'

print("Running DistilBERT sentiment inference (this takes ~10-15 min)...")
for i in tqdm(range(0, len(triggered_clauses), BATCH_SIZE)):
    batch = triggered_clauses[i:i+BATCH_SIZE]
    batch_truncated = [c[:512] for c in batch]
    results = sentiment_model(batch_truncated, batch_size=BATCH_SIZE)
    for clause, result in zip(batch, results):
        label = result[0]['label'].lower()
        sentiment_cache[clause] = label

print(f"\nDone! Scored {len(sentiment_cache):,} unique clauses ✅")
pos_count = sum(1 for v in sentiment_cache.values() if v == 'positive')
neg_count = sum(1 for v in sentiment_cache.values() if v == 'negative')
print(f"  Positive: {pos_count:,} ({pos_count/len(sentiment_cache)*100:.1f}%)")
print(f"  Negative: {neg_count:,} ({neg_count/len(sentiment_cache)*100:.1f}%)")


Unique triggered clauses to score: 54,311
Running DistilBERT sentiment inference (this takes ~10-15 min)...


100%|██████████| 849/849 [29:20<00:00,  2.07s/it]


Done! Scored 54,311 unique clauses ✅
  Positive: 31,206 (57.5%)
  Negative: 23,105 (42.5%)


In [ ]:
# Step 9: Apply cached sentiments to full dataset and Export
print("Applying cached sentiments to full dataset...")

rows_final = []
for idx, row in clause_df.iterrows():
    labels = {}
    for attr in attrs:
        # If no trigger keyword was found, it's not mentioned
        if not row[f'trigger_{attr}']:
            labels[attr] = 'not_mentioned'
        else:
            # Otherwise, use the model's sentiment for that clause
            labels[attr] = sentiment_cache.get(row['clause'], 'not_mentioned')
            
    rows_final.append({
        'review_idx': row['review_idx'],
        'rating': row['rating'],
        'clause': row['clause'],
        **labels
    })

labeled_df = pd.DataFrame(rows_final)

import os
os.makedirs('data', exist_ok=True)
labeled_df.to_csv('data/labeled_clauses.csv', index=False)
print("Exported labeled data to data/labeled_clauses.csv ✅")

# Print Final Distribution
print("\n=== FINAL LABEL DISTRIBUTION ===")
print(f"{'Attribute':<20} {'positive':>10} {'negative':>10} {'total_labeled':>15}")
print("-" * 57)
for attr in attrs:
    counts = labeled_df[attr].value_counts()
    pos = counts.get('positive', 0)
    neg = counts.get('negative', 0)
    print(f"{attr:<20} {pos:>10,} {neg:>10,} {pos+neg:>15,}")


Applying cached sentiments to full dataset...
Exported labeled data to data/labeled_clauses.csv ✅

=== FINAL LABEL DISTRIBUTION ===
Attribute              positive   negative   total_labeled
---------------------------------------------------------
cleanliness               6,060      3,529           9,589
staff_service            13,845     10,519          24,364
wifi_quality                913      1,358           2,271
noise_level               3,044      4,211           7,255
location                 11,782      5,675          17,457


: 